# NB-12-AVAILABILITY · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-12-AVAILABILITY**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-12-AVAILABILITY.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This is an observed-data audit of the volume’s historical-information rule: a stored value is not necessarily a forecast that was available before delivery. It audits availability rather than forecasting skill.

- [Volume V, §6.6 — Publication time and receipt time](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c6-6)
- [Volume VI, Chapter 8 — Forecast evaluation and external validation](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c8)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-12-AVAILABILITY', require_key=True)
print(setup_status)


# NB-12-AVAILABILITY — Does a forecast exist before delivery?
**Question:** what evidence distinguishes a stored forecast value from a value
available at a historical decision time? This companion to NB-12 replays real
EWB responses retained on 15 October 2026. It is an availability audit, not a
forecast-accuracy comparison. No synthetic data is created; calculations call the authenticated private methods API.
Prerequisites: target time, run time, ingestion, publication, and information
cutoffs. We reject post-delivery power records, inspect a historical GFS sample,
apply an explicit local timestamp policy, and preserve failed service checks.
Reruns require the thin client, authenticated private service and retained inputs.

In [ ]:
NOTEBOOK_ID = 'NB-12-AVAILABILITY'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

from collections import Counter
def dt(s):return datetime.fromisoformat(s.replace('Z','+00:00'))
cutoff=dt(inputs['cutoff'])
print('Local raw weather inspection:',inputs['weather'][:1])
audits={name:execute(name,'availability_audit',request)['availability_ledger'] for name,request in inputs['requests'].items()}
weather=inputs['weather'];selected_ids=[];eligible_ids=[]
for name,indices in inputs['groups'].items():
    parents=[name+':published_time',name+':received_time']
    allowed=[{row['assertion_id'] for row in audits[parent]['records'] if row['status']=='available'} for parent in parents]
    candidates=allowed[0]&allowed[1]
    assertions=[]
    for index in indices:
        r=weather[index];ident='weather:'+str(index)
        if ident not in candidates:continue
        # The eight nulls remain in raw data/audits. If a null ever becomes eligible,
        # this finite selector profile must refuse rather than drop or fill it.
        assert r['value'] is not None,'Eligible missing value requires a separate profile'
        assertions.append(dict(assertion_id=ident,target_id=r['zone_id']+':'+r['variable'],
            valid_time=r['ts_utc'],issue_time=r['run_time_utc'],published_time=r['published_ts_utc'],received_time=r['ingest_ts_utc'],
            value=r['value'],unit='UNVERIFIED_SOURCE_UNIT:'+r['variable'],source_artifact_sha256=inputs['weather_source_sha256']))
    request=dict(inputs=dict(forecast_assertions=assertions,realization_assertions=[],
        information_cutoff=dict(clock='received_time',timestamp=inputs['cutoff'])),parameters=dict(variant='select'),
        context=dict(input_origin='NB-12-AVAILABILITY: recorded publication AND ingestion cutoff then latest eligible issue; weather units/source issuance unverified'))
    result=execute(name+':select','forecast_vintage_transform',request,parents)['selected_assertions']
    eligible_ids += [int(r['assertion_id'].split(':')[1]) for r in result['selection_ledger'] if r['eligible']]
    for row in result['records']:
        index=int(row['assertion_id'].split(':')[1]);assert row['value']==weather[index]['value']
        selected_ids.append(index)
assert len(set(selected_ids))==len(selected_ids) and len(set(eligible_ids))==len(eligible_ids)
eligible=[weather[i] for i in sorted(eligible_ids)]
selected=sorted([weather[i] for i in selected_ids],key=lambda r:(r['ts_utc'],r['zone_id'],r['variable']))
f=dict(power=inputs['power'],weather=weather,eligible=eligible,selected=selected,controls=inputs['controls'],audit=dict(inputs['audit']))
f['audit'].update(selected_missing_values=sum(r['value'] is None for r in selected),eligible_rows=len(eligible),selected_rows=len(selected),
    selected_lead_min=min(r['lead_hours'] for r in selected),selected_lead_max=max(r['lead_hours'] for r in selected))
for name,result in audits.items():
    assert result['complete_point_in_time_reconstruction'] is False and result['source_claims_verified'] is False
    if name.startswith(('generation-forecast','load-forecast')):
        source={r['assertion_id']:r for r in inputs['requests'][name]['inputs']['assertion_provenance']}
        clock=inputs['requests'][name]['inputs']['information_cutoff']['clock']
        assert all(r['status']==('unknown' if source[r['assertion_id']][clock] is None else 'unavailable') for r in result['records'])
print(json.dumps(f['power'],indent=2));print(json.dumps(f['audit'],indent=2))

## 1. The word “forecast” cannot repair a clock
For delivery on 1 January 2019, all 842 generation and 148 load records have
negative leads and run timestamps after delivery. Their run-time bases explicitly
identify post-delivery publication or ingestion. Forecast-type labels such as
day_ahead are not evidence of issue time. The two retained as-of queries at
31 December 2018 12:00 UTC return zero rows. These observations are real negative
controls: they cannot be used as pre-delivery features in this sample.
An empty cutoff response is not independent proof of a correct implementation
of every as-of view. We have not reconstructed an earlier source archive.

In [ ]:
for name,a in f['power'].items():
    assert a['rows']==a['post_delivery']==a['negative_leads'] and a['cutoff_rows']==0
    print(name, 'rejected pre-delivery rows:',a['rows'])
plt=setup_plot();fig,ax=plt.subplots(figsize=(9,4))
bars=ax.bar(['Generation','Load'],[842,148],color=['#BF616A','#EBCB8B'])
ax.bar_label(bars,padding=4,color='#ECEFF4');ax.set_ylim(0,950)
ax.set(ylabel='Observed records',title='2019 delivery: every sampled power run is post-delivery')
fig.tight_layout();plt.show()

## 2. Weather contains a candidate, with missing values and late arrivals
The complete 1 May 2026 GFS query has 2,144 rows, covering DE_LU and GB,
four variables and 64 target-time/zone/variable keys. Leads from 0 to 240 hours
reconcile exactly with target minus run time; target/run keys are unique.
Unlike the power sample, many runs precede delivery. But 160 records were
ingested after delivery and 64 were published after delivery. Run time alone
cannot establish availability. These counts overlap and must not be added.

Eight ssrd values are null. They remain missing, not zeros. None survives the
local cutoff used below. The missing descriptor leaves units, accumulated versus
instantaneous values, and timestamp provenance unresolved. Therefore we plot
counts and lead times, not physical weather levels or forecast errors.

In [ ]:
print('Missing values by variable:',dict(Counter(r['variable'] for r in f['weather'] if r['value'] is None)))
assert f['audit']['missing_weather_values']==8
assert f['audit']['selected_missing_values']==0
fig,ax=plt.subplots(figsize=(10,4))
bars=ax.bar(['All rows','Ingested after delivery','Published after delivery','Missing values'],
    [2144,f['audit']['postdelivery_ingest'],f['audit']['postdelivery_publication'],8],color=['#88C0D0','#BF616A','#EBCB8B','#B48EAD'])
ax.bar_label(bars,padding=4,color='#ECEFF4');ax.set_ylim(0,2450)
ax.set(ylabel='Records (overlapping categories)',title='Clock and missingness checks on retained weather records')
fig.tight_layout();plt.show()

## 3. A conservative local policy, explicitly conditional
Set the cutoff to 30 April 2026 12:00 UTC. Require **all three** recorded times—
run, ingest, and publication—to be non-null and no later than the cutoff.
This retains 1,984 rows and all 64 target keys. Select the greatest run time
within each target/zone/variable group; fail on tied rows instead of arbitrarily
choosing a revision. This yields 64 unique, finite selected values.

The selection is reproducible arithmetic on recorded metadata. It does not
establish that publication times are authentic source issuance, or replace a
working API as-of contract. We call these **locally eligible**, not verified
historically available forecasts. No truth data or forecast scores are supplied.

In [ ]:
print('Local cutoff:',cutoff.isoformat())
assert len(f['eligible'])==1984 and len(f['selected'])==64
for r in f['selected']:
    assert all(dt(r[k])<=cutoff for k in ('run_time_utc','ingest_ts_utc','published_ts_utc'))
print('Selected lead range (hours):',f['audit']['selected_lead_min'],f['audit']['selected_lead_max'])
print('Selected runs:',dict(Counter(r['run_time_utc'] for r in f['selected'])))
fig,ax=plt.subplots(figsize=(10,4))
for zone,color in [('DE_LU','#88C0D0'),('GB','#EBCB8B')]:
    rr=[r for r in f['selected'] if r['zone_id']==zone and r['variable']=='t2m']
    ax.plot([dt(r['ts_utc']) for r in rr],[r['lead_hours'] for r in rr],marker='o',label=zone,color=color)
ax.set(ylabel='Selected run lead (hours)',xlabel='Target time UTC',title='Locally eligible temperature runs: metadata only');ax.legend();ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 4. Failed service checks remain failed
The retained catalog lists generation, load and GFS routes, but their descriptor
and coverage calls returned 404. Query responses identify snapshot
`20260505T043528Z`, while the catalog names energy snapshot `20260709T091009Z`.
This is a serving-contract inconsistency; catalog presence is not readiness.

GFS as-of timed out for the full day and again for one timestamp and one zone.
That is evidence of two failed requests, not proof that the route always fails.
All three publication-as-of calls returned HTTP 400 asking for `published_as_of`;
the connector exposes `as_of` but no dedicated `published_as_of` argument.
Latest queries for 10 October returned zero rows for these three routes. Those
bounded empty results do not establish empty full histories. The audit replays
these retained October 15 outcomes; it does not assert current service health.

In [ ]:
for name,status in f['controls'].items():print(name,':',status)
for name in ('silver_power_generation_forecast_vintages','silver_power_load_forecast_vintages','silver_weather_gfs_forecast'):
    assert inputs['retained_responses']['desc_'+name]['isError']
    assert inputs['retained_responses']['cov_'+name]['isError']
print('Publication cutoff: not validated. Server weather cutoff: not validated.')

## Exercise and next evidence
Why not backdate a record from its day-ahead label? That invents knowledge time.
Why not use the newest run regardless of ingestion? A model run may have existed
before delivery while this data pipeline acquired it afterward. Why not drop
missing ssrd values as zeros? That changes the observation rather than its mask.

An availability-audit notebook is justified; a predictive evaluation is gated.
Required next evidence: repaired descriptors and snapshot routing, documented
source clocks and variable units, working as-of parameters, repeated delivery
windows, and independently aligned realizations. A later source repair should
produce a new capture and comparison, not overwrite these failed controls.
Source: retained EWB catalog, descriptors, errors and complete query responses.

In [ ]:
bad=copy.deepcopy(next(item['request'] for item in execution_plan if item['method']=='forecast_vintage_transform'))
bad['inputs']['forecast_assertions'].append(copy.deepcopy(bad['inputs']['forecast_assertions'][0]))
reject('forecast_vintage_transform',bad)
bad=copy.deepcopy(next(item['request'] for item in execution_plan if item['method']=='forecast_vintage_transform'))
bad['inputs']['forecast_assertions'][0]['received_time']=None
reject('forecast_vintage_transform',bad)
(session.output/'consumer-summary.json').write_text(json.dumps(dict(power=f['power'],audit=f['audit'],selected=f['selected'],eligible=f['eligible'],controls=f['controls']),indent=2,allow_nan=False)+'\n')
summary=session.finish({'observed_only':True,'postdelivery_power_rejected':True,'local_cutoff_conditional':True,
    'missing_values_preserved':True,'service_failures_retained':True,'new_synthetic_data':False})